# Figure 3 — Generalization Switching Time

**Goal:** Identify when generalization arises during the flow.

We construct a **hybrid model**: follow $\hat{u}^\star$ on $[0, \tau]$, then $u_\theta$ on $[\tau, 1]$.

- For $\tau = 1$: full closed-form trajectory → memorization.
- For $\tau = 0$: full learned trajectory → generalization.
- The **switching time** is the threshold $\tau$ where the model transitions from generalization to memorization.

In [ ]:
import os, sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/YOUR_REPO/ClosedFlowMatching.git 2>/dev/null || true
    os.chdir('ClosedFlowMatching')
    !pip install -q -r requirements.txt
    sys.path.insert(0, '.')
else:
    sys.path.insert(0, '..')

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from src.data.toy import ToyDataset
from src.models.mlp import VelocityMLP
from src.flow_matching.cfm import CFMTrainer
from src.flow_matching.sampler import ode_sample, ode_sample_hybrid
from src.metrics.evaluation import nearest_neighbor_distance

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False
})
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 3.1 Train a flow matching model

In [ ]:
ds = ToyDataset('gaussian_mixture', n_samples=2000, seed=42)
data = ds.data.to(device)

model = VelocityMLP(data_dim=2, hidden_dim=256, n_layers=4)
trainer = CFMTrainer(model, lr=1e-3, device=device)
losses = trainer.train(ds, n_steps=20000, batch_size=256, log_every=5000)

print('Training done.')

## 3.2 Hybrid sampling: vary $\tau$ and measure memorization

We fix 256 initial noise vectors and, for each $\tau$, generate samples with the hybrid model.
We measure the mean nearest-neighbor distance to the training set.

In [ ]:
n_gen = 256
torch.manual_seed(0)
x0_fixed = torch.randn(n_gen, 2, device=device)

tau_values = np.linspace(0.0, 1.0, 21)
mean_nn_dists = []

for tau in tau_values:
    gen = ode_sample_hybrid(model, data, x0_fixed, tau=tau, n_steps=200, device=device).cpu()
    dists = nearest_neighbor_distance(gen, ds.data)
    mean_nn_dists.append(dists.mean().item())

# Also get the baseline: pure u_theta
gen_pure = ode_sample(model, n_gen, (2,), n_steps=200, device=device).cpu()
baseline_dist = nearest_neighbor_distance(gen_pure, ds.data).mean().item()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tau_values, mean_nn_dists, 'o-', color='steelblue', linewidth=2, label='Hybrid model')
ax.axhline(y=baseline_dist, color='gray', linestyle='--', alpha=0.7, label=r'Pure $u_\theta$')
ax.set_xlabel(r'Switching time $\tau$')
ax.set_ylabel('Mean NN distance to train')
ax.set_title('Generalization switching time')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_switching_time.pdf', bbox_inches='tight')
plt.show()

## 3.3 Visualize generated samples for various $\tau$

In [ ]:
tau_show = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
fig, axes = plt.subplots(1, len(tau_show), figsize=(3.5 * len(tau_show), 3.5))

for i, tau in enumerate(tau_show):
    gen = ode_sample_hybrid(model, data, x0_fixed[:200], tau=tau, n_steps=200, device=device).cpu().numpy()
    ax = axes[i]
    ax.scatter(ds.data[:, 0], ds.data[:, 1], s=2, alpha=0.15, c='steelblue')
    ax.scatter(gen[:, 0], gen[:, 1], s=5, alpha=0.6, c='tomato')
    ax.set_title(f'$\\tau$ = {tau:.1f}')
    ax.set_aspect('equal')
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)

plt.tight_layout()
plt.savefig('fig3_scatter_tau.pdf', bbox_inches='tight')
plt.show()

## 3.4 Switching time vs dataset geometry

We repeat the hybrid experiment for different toy datasets.

In [ ]:
ds_names = ['moons', 'rings', 'gaussian_mixture']
tau_values = np.linspace(0.0, 1.0, 21)

fig, ax = plt.subplots(figsize=(8, 5))
colors_geom = ['steelblue', 'seagreen', 'coral']

for j, ds_name in enumerate(ds_names):
    ds_g = ToyDataset(ds_name, n_samples=2000, seed=42)
    data_g = ds_g.data.to(device)

    model_g = VelocityMLP(data_dim=2, hidden_dim=256, n_layers=4)
    trainer_g = CFMTrainer(model_g, lr=1e-3, device=device)
    trainer_g.train(ds_g, n_steps=20000, batch_size=256, log_every=20000)

    x0_g = torch.randn(n_gen, 2, device=device)
    nn_dists = []
    for tau in tau_values:
        gen = ode_sample_hybrid(model_g, data_g, x0_g, tau=tau, n_steps=200, device=device).cpu()
        d_nn = nearest_neighbor_distance(gen, ds_g.data).mean().item()
        nn_dists.append(d_nn)

    ax.plot(tau_values, nn_dists, 'o-', color=colors_geom[j], linewidth=2, label=ds_name)

ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Mean NN distance')
ax.set_title('Switching time vs dataset geometry')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_geometry_comparison.pdf', bbox_inches='tight')
plt.show()

## 3.5 Switching time vs dimension

In [ ]:
dims_switch = [2, 10, 50, 100]
tau_values = np.linspace(0.0, 1.0, 21)

fig, ax = plt.subplots(figsize=(8, 5))
colors_dim = cm.viridis(np.linspace(0.15, 0.95, len(dims_switch)))

for k, d in enumerate(dims_switch):
    ds_d = ToyDataset('gaussian_mixture_nd', n_samples=2000, dim=d, seed=42)
    data_d = ds_d.data.to(device)

    model_d = VelocityMLP(data_dim=d, hidden_dim=256, n_layers=4)
    trainer_d = CFMTrainer(model_d, lr=1e-3, device=device)
    trainer_d.train(ds_d, n_steps=20000, batch_size=256, log_every=20000)

    x0_d = torch.randn(n_gen, d, device=device)
    nn_dists = []
    for tau in tau_values:
        gen = ode_sample_hybrid(model_d, data_d, x0_d, tau=tau, n_steps=200, device=device).cpu()
        d_nn = nearest_neighbor_distance(gen, ds_d.data).mean().item()
        nn_dists.append(d_nn)

    ax.plot(tau_values, nn_dists, 'o-', color=colors_dim[k], linewidth=2, label=f'd = {d}')

ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Mean NN distance')
ax.set_title('Switching time vs dimension')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_dimension_switching.pdf', bbox_inches='tight')
plt.show()

## 3.6 Switching time vs model capacity

In [ ]:
hidden_dims = [32, 64, 128, 256, 512]
tau_values = np.linspace(0.0, 1.0, 21)
ds_cap = ToyDataset('gaussian_mixture', n_samples=2000, seed=42)
data_cap = ds_cap.data.to(device)

fig, ax = plt.subplots(figsize=(8, 5))
colors_cap = cm.plasma(np.linspace(0.15, 0.85, len(hidden_dims)))

for k, hd in enumerate(hidden_dims):
    model_c = VelocityMLP(data_dim=2, hidden_dim=hd, n_layers=4)
    trainer_c = CFMTrainer(model_c, lr=1e-3, device=device)
    trainer_c.train(ds_cap, n_steps=20000, batch_size=256, log_every=20000)

    x0_c = torch.randn(n_gen, 2, device=device)
    nn_dists = []
    for tau in tau_values:
        gen = ode_sample_hybrid(model_c, data_cap, x0_c, tau=tau, n_steps=200, device=device).cpu()
        d_nn = nearest_neighbor_distance(gen, ds_cap.data).mean().item()
        nn_dists.append(d_nn)

    n_params = sum(p.numel() for p in model_c.parameters())
    ax.plot(tau_values, nn_dists, 'o-', color=colors_cap[k], linewidth=2,
            label=f'hidden={hd} ({n_params/1000:.0f}K params)')

ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Mean NN distance')
ax.set_title('Switching time vs model capacity')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('fig3_capacity_switching.pdf', bbox_inches='tight')
plt.show()

## 3.7 Switching time vs training duration

In [ ]:
training_steps_list = [1000, 5000, 10000, 20000, 50000]
tau_values = np.linspace(0.0, 1.0, 21)
ds_tr = ToyDataset('gaussian_mixture', n_samples=2000, seed=42)
data_tr = ds_tr.data.to(device)

fig, ax = plt.subplots(figsize=(8, 5))
colors_tr = cm.coolwarm(np.linspace(0.1, 0.9, len(training_steps_list)))

for k, n_st in enumerate(training_steps_list):
    model_t = VelocityMLP(data_dim=2, hidden_dim=256, n_layers=4)
    trainer_t = CFMTrainer(model_t, lr=1e-3, device=device)
    trainer_t.train(ds_tr, n_steps=n_st, batch_size=256, log_every=n_st)

    x0_t = torch.randn(n_gen, 2, device=device)
    nn_dists = []
    for tau in tau_values:
        gen = ode_sample_hybrid(model_t, data_tr, x0_t, tau=tau, n_steps=200, device=device).cpu()
        d_nn = nearest_neighbor_distance(gen, ds_tr.data).mean().item()
        nn_dists.append(d_nn)

    ax.plot(tau_values, nn_dists, 'o-', color=colors_tr[k], linewidth=2, label=f'{n_st} steps')

ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Mean NN distance')
ax.set_title('Switching time vs training duration')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_training_duration.pdf', bbox_inches='tight')
plt.show()